# NB25 Econometric tests for bubbles — Chapter 25

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- The only package outside the standard library is `numpy` (run and checked on Python 3.13 with numpy 2.5.4). There is no install cell; if it is missing, run `pip install numpy==2.5.4`.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB25-1: PWY/PSY tests on synthetic data — computes SADF and GSADF on a fake price series with a planted explosive interval (a demonstration of method; no actual market price data are used)
import numpy as np

rng = np.random.default_rng(25)      # fixed seed — the same results are reproduced

def make_series(T=200, bubble=(100, 130), beta=1.04, sigma=1.0, collapse_to=None):
    """A driftless random walk in which only the interval [bubble[0], bubble[1]) grows as y_t = beta·y_{t-1} + e_t; when the interval ends, the series drops back to near its level before the interval."""
    y = np.empty(T); y[0] = 30.0
    e = rng.normal(0.0, sigma, T)
    for t in range(1, T):
        if bubble[0] <= t < bubble[1]:
            y[t] = beta * y[t - 1] + e[t]
        elif t == bubble[1]:
            y[t] = (collapse_to if collapse_to is not None else y[bubble[0] - 1]) + e[t]
        else:
            y[t] = y[t - 1] + e[t]
    return y

def adf_grid(y, min_win):
    """Computes at once the lag-0 ADF t statistic for every window [s, e] (Δy_t regressed on a constant and y_{t-1}; the t ratio of the y_{t-1} coefficient).
    Returns: a (T−1)×(T−1) array A, where A[s, e] is the t of the regression whose window uses Δy for observations s+1 … e+1. Windows that are too short are NaN."""
    x = y[:-1]; z = np.diff(y)                     # x_t = y_{t-1}, z_t = Δy_t, t = 1 … T−1
    n = len(z)
    def ps(a):                                    # prefix sums with a leading 0 — any range sum in one subtraction
        return np.concatenate(([0.0], np.cumsum(a)))
    Px, Pz, Pxx, Pxz, Pzz = ps(x), ps(z), ps(x * x), ps(x * z), ps(z * z)
    s = np.arange(n)[:, None]; e = np.arange(n)[None, :]
    cnt = (e - s + 1).astype(float)
    Sx = Px[e + 1] - Px[s]; Sz = Pz[e + 1] - Pz[s]
    Sxx = Pxx[e + 1] - Pxx[s]; Sxz = Pxz[e + 1] - Pxz[s]; Szz = Pzz[e + 1] - Pzz[s]
    with np.errstate(invalid="ignore", divide="ignore"):
        den = cnt * Sxx - Sx * Sx
        beta = (cnt * Sxz - Sx * Sz) / den
        alpha = (Sz - beta * Sx) / cnt
        rss = Szz - alpha * Sz - beta * Sxz
        s2 = rss / (cnt - 2)
        t = beta / np.sqrt(s2 * cnt / den)
    t[(cnt < min_win) | (e < s)] = np.nan
    return t

def sadf_gsadf(y, r0):
    T = len(y); w = int(np.floor(r0 * T))
    A = adf_grid(y, w)
    sadf = np.nanmax(A[0, :])                      # start fixed (0), end extended
    bsadf = np.nanmax(A, axis=0)                   # for each end r2, the maximum over all starts — for date-stamping
    gsadf = np.nanmax(bsadf)                       # the maximum over all (r1, r2)
    return sadf, gsadf, bsadf, A

T = 200
r0 = 0.01 + 1.8 / np.sqrt(T)                       # PSY's minimum-window rule → 0.137 (T = 200)
y = make_series(T)
sadf, gsadf, bsadf, A = sadf_gsadf(y, r0)
PSY_T200 = {"SADF 95%": 1.40, "SADF 99%": 1.90, "GSADF 95%": 2.08, "GSADF 99%": 2.70}   # PSY (2015) Table 1, finite sample, T=200, r0=0.137
print(f"Synthetic data: T = {T}, explosive interval t ∈ [100, 130), β = 1.04, then a collapse. Minimum window r0 = {r0:.3f} → {int(np.floor(r0*T))} observations")
print(f"SADF  = {sadf:.3f}   (PSY Table 1 critical values 95% {PSY_T200['SADF 95%']:.2f}, 99% {PSY_T200['SADF 99%']:.2f})")
print(f"GSADF = {gsadf:.3f}   (PSY Table 1 critical values 95% {PSY_T200['GSADF 95%']:.2f}, 99% {PSY_T200['GSADF 99%']:.2f})")
print(f"Single ADF on the full sample (start 0, end T) = {A[0, -1]:.3f}  ← the explosive interval is buried in the long unit-root stretch and does not show")
peak = int(np.nanargmax(bsadf)) + 1
print(f"Endpoint where BSADF is largest: t = {peak} (just before 130, the end of the planted interval, means the test caught it well)")

Synthetic data: T = 200, explosive interval t ∈ [100, 130), β = 1.04, then a collapse. Minimum window r0 = 0.137 → 27 observations
SADF  = 11.918   (PSY Table 1 critical values 95% 1.40, 99% 1.90)
GSADF = 12.361   (PSY Table 1 critical values 95% 2.08, 99% 2.70)
Single ADF on the full sample (start 0, end T) = -2.466  ← the explosive interval is buried in the long unit-root stretch and does not show
Endpoint where BSADF is largest: t = 129 (just before 130, the end of the planted interval, means the test caught it well)


In [ ]:
# NB25-2: Monte Carlo critical values and date-stamping — generates many null (random walk) series to get the 95% and 99% quantiles of GSADF and the sequence of BSADF critical values, then stamps dates on the synthetic data (a demonstration of method)
def mc_critical_values(T, r0, reps=2000, seed=2025):
    g = np.random.default_rng(seed)
    w = int(np.floor(r0 * T))
    gs = np.empty(reps); bs = np.empty((reps, T - 1))
    for i in range(reps):
        yn = 30.0 + np.cumsum(g.normal(0.0, 1.0, T))           # null: driftless random walk
        A = adf_grid(yn, w)
        bs[i] = np.nanmax(A, axis=0); gs[i] = np.nanmax(bs[i])
    return gs, bs

gs_null, bs_null = mc_critical_values(T, r0)
cv95, cv99 = np.quantile(gs_null, [0.95, 0.99])
print(f"Null random walks, {len(gs_null)} replications → GSADF critical values 95% {cv95:.2f}, 99% {cv99:.2f}   (PSY Table 1: 2.08, 2.70 — this cell: 2,000 replications, lag 0, no drift; PSY: weak drift d=η=1)")
print(f"GSADF of the synthetic data {gsadf:.3f} → at 95%: {'reject (explosive)' if gsadf > cv95 else 'fail to reject'}, at 99%: {'reject' if gsadf > cv99 else 'fail to reject'}")

# date-stamping: for each endpoint t, use the 95% quantile of the null BSADF_t as the critical value (PSY use the SADF critical value based on ⌊T r2⌋ observations; here a sequence taken directly from the same Monte Carlo stands in for it)
bs_cv95 = np.nanquantile(bs_null, 0.95, axis=0)
LT = int(np.ceil(np.log(T)))                      # minimum duration log(T) ≈ 5.3 → 6 observations
flag = bsadf > bs_cv95
episodes, start = [], None
for t, f in enumerate(flag):
    if f and start is None: start = t
    if (not f or t == len(flag) - 1) and start is not None:
        end = t if not f else t + 1
        if end - start >= LT: episodes.append((start + 1, end + 1))   # +1: Δy index → time in the original series
        start = None
print(f"Explosive intervals stamped with the critical-value sequence (endpoint-wise BSADF 95%; minimum duration {LT} observations): {episodes}")
print(f"Planted interval: [100, 130). The start is usually caught a few observations late and the end just after the collapse — the test sees 'growing now', not 'has started to grow'.")
print("An interval caught outside the planted one is a false positive — allow 5% at every endpoint and somewhere among the 170-odd endpoints one crosses by chance (the multiplicity issue Oldani et al. describe).")

Null random walks, 2000 replications → GSADF critical values 95% 2.11, 99% 2.65   (PSY Table 1: 2.08, 2.70 — this cell: 2,000 replications, lag 0, no drift; PSY: weak drift d=η=1)
GSADF of the synthetic data 12.361 → at 95%: reject (explosive), at 99%: reject
Explosive intervals stamped with the critical-value sequence (endpoint-wise BSADF 95%; minimum duration 6 observations): [(49, 63), (102, 130)]
Planted interval: [100, 130). The start is usually caught a few observations late and the end just after the collapse — the test sees 'growing now', not 'has started to grow'.
An interval caught outside the planted one is a false positive — allow 5% at every endpoint and somewhere among the 170-odd endpoints one crosses by chance (the multiplicity issue Oldani et al. describe).


In [ ]:
# NB25-3: size distortion under time-varying volatility, and the wild bootstrap — on null data with no bubble, where only the variance rises, how often do textbook critical values say "bubble"? (a demonstration of method)
def null_with_vol_break(T, g, s1=1.0, s2=3.0, brk=None):
    brk = T // 2 if brk is None else brk
    sig = np.where(np.arange(T) < brk, s1, s2)
    return 30.0 + np.cumsum(g.normal(0.0, 1.0, T) * sig)

g = np.random.default_rng(2526); reps = 400; w = int(np.floor(r0 * T))
rej_const = rej_break = 0
for i in range(reps):
    y0 = 30.0 + np.cumsum(g.normal(0.0, 1.0, T))                       # constant variance
    y1 = null_with_vol_break(T, g)                                      # variance 9x (standard deviation 3x) from the middle
    rej_const += np.nanmax(adf_grid(y0, w)) > cv95
    rej_break += np.nanmax(adf_grid(y1, w)) > cv95
print(f"{reps} null series of each kind, Monte Carlo 95% critical value {cv95:.2f} — nominal size 5%")
print(f"  constant variance:            rejection rate {rej_const / reps:.1%}")
print(f"  standard deviation 3x midway: rejection rate {rej_break / reps:.1%}   ← with no bubble at all, it says 'bubble' this often")

# wild bootstrap (the skeleton of the Hafner and Phillips–Shi approach): multiply the residuals of the observed data by ±1 or normal weights to rebuild null samples, and take the critical value from them — the change of variance over time is preserved as it is
def wild_bootstrap_cv(y, w, B=199, seed=7):
    gb = np.random.default_rng(seed)
    resid = np.diff(y) - np.mean(np.diff(y))                            # residuals under the null (unit root): Δy minus its mean
    stats = np.empty(B)
    for b in range(B):
        eb = resid * gb.normal(0.0, 1.0, len(resid))                    # wild: multiply each residual by an independent N(0,1)
        yb = y[0] + np.concatenate(([0.0], np.cumsum(eb)))
        stats[b] = np.nanmax(adf_grid(yb, w))
    return np.quantile(stats, 0.95)

# pick one null series that textbook critical values would call a "bubble" (trying seeds from 0 in order — reproducible)
for sd in range(200):
    y_break = null_with_vol_break(T, np.random.default_rng(sd))
    stat = np.nanmax(adf_grid(y_break, w))
    if stat > cv95: break
cv_wb = wild_bootstrap_cv(y_break, w)
print(f"\nOne null series with changing variance (seed {sd}): GSADF {stat:.2f}. Textbook critical value {cv95:.2f} → {'reject' if stat > cv95 else 'fail to reject'}; wild bootstrap 95% critical value {cv_wb:.2f} → {'reject' if stat > cv_wb else 'fail to reject'}")
print("The bootstrap critical value moves with the volatility structure of the data. The same statistic can be 'bubble' under one critical value and 'not a bubble' under another.")

400 null series of each kind, Monte Carlo 95% critical value 2.11 — nominal size 5%
  constant variance:            rejection rate 4.8%
  standard deviation 3x midway: rejection rate 34.0%   ← with no bubble at all, it says 'bubble' this often

One null series with changing variance (seed 0): GSADF 2.32. Textbook critical value 2.11 → reject; wild bootstrap 95% critical value 3.45 → fail to reject
The bootstrap critical value moves with the volatility structure of the data. The same statistic can be 'bubble' under one critical value and 'not a bubble' under another.
